<b> Project Overview</b><br>

Business problem draft<br>
Intended user or decision context<br>
Dataset description and source link<br> [x]
List of raw files or tables<br>
Explanation of what one row means in each important table<br>
Initial row and column counts<br>
Data types<br>
Missing values<br>
Duplicate records or duplicate keys<br>
Suspicious, impossible, or confusing values<br>
Initial cleaning plan<br>
Potential target or outcome variable<br>
Early ethics, privacy, governance, or responsible-use concerns<br>


---
## Loading The Data <br>

The initial step in this is project is to load the required libraries and the datasets. 
 
The libraries required are...   
        - pathlib : a file handler that treats paths as objects rather than text strings.   
        - pandas : a library of functions that are used to manipulate dataframes.  
        - numpy : a library of math functions.  

Next, we define the root folder where the project folders and files will be saved.  

The project directory contains the following folders...  
        - The data folder : This folder contains raw data files.  
        - The output folder : This folder contains the final ABT and processed data files.    
        - The report folder : This folder contains html reports, graphs, and analysis.  



In [1]:
# This code loads libraries and sets directors for the project.
#  pathlib is a file handler. treats paths as objects not strings
# pandas brings in df functions
# numpy provides math functions.

#Checking my Python environment 
import sys; print(sys.executable); print(sys.version)

from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

#Set repository root to the current working directory. One step above readme location.
display(REPO_ROOT)

#Create directories for data, outputs, and reports if they don't exist.
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

#set max column display to 100 rows for pandas dataframes.
pd.set_option("display.max_columns", 100)

/usr/bin/python3
3.13.5 (main, Aug 10 2026, 12:06:59) [GCC 14.2.0]


PosixPath('/workspaces/Project-Option-1-Matt-Flores')

---               
## Dataset Review

Now that the data is loaded we can review and profile each table loaded into the directory.  
There are 5 raw data files loaded into the data folder.
1) **movies** : This is the main df because the primary key, ***movieId***, in this df is the project's **unit of analysis**.  

2) **ratings** : This dataset contains almost 100k reviews throughout 1995 to 2016. This data will need to be aggregated to be merged with the movies table. *Each row in this dataset represents a single user's rating for a movie. There are multiple ratings per movie.*  

3) **links** : Contains the imdbId & tmdbId associated with each movieId. It allows the ratings df to be joined with the credits and keywords tables without having to cross refence the movies dataset, which contains all 3 id types. *Each row in this dataset represents a movieId and related links. Each movieId should have 2 additional Ids.*  

4) **credits** : Lists the directors and top billed actors in each film. Maybe there is a correlation between revenue and directors/actors. *Each row in this dataset represents a movieID with associated directions/actors.*  

5) **keywords** : Contains The Movie Database ID along with a multi-valued field listing descriptive words for each movie. Genre is a broad categorization of a film. keywords df can provide more detail on themes, locations, movie plots, characters, etc. *Each row in this dataset represents a single movie.*

In [2]:
#This code loads the raw data into dataframes. And creates a column summary by table df.
# Load the datasets
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

# Create a dictionary (keys paired with values) to hold the DataFrames for easier access. We can loop through tables, etc.
tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

print("Datasets loaded into dataframes. Dictionary of dataframes called 'tables' created.")

Datasets loaded into dataframes. Dictionary of dataframes called 'tables' created.


---               
## Data Structure

The first step in data profiling is reviewing the data structure. Data stucture consists of rows and columns. To analyze data we must first understand what each row of the data represents and the attributes (columns) grouped with each row.

Instead of running shape() for each table I thought it would be best to aggregate each table's structure into summary table.  

I appended table name along with the associated columns into a single dataframe.  
I repeated the process a second time to gather the shape info from each table.

In [3]:
# Create a blank list called column_summary that we can populate.
column_summary = []

# Populate the list using a loop that appends table names and each column within that table.
for name, df in tables.items():
    column_summary.append({"table_name": name, **{f"col_{i+1}": col for i, col in enumerate(df.columns)}})

# Build a summary DataFrame that shows the table name and each column name in separate columns.
# Converted summary_data list into a pd.dataframe and set index equal to table name.
column_summary_df = pd.DataFrame(column_summary).set_index("table_name")

# Display the result
display(column_summary_df)

raw_data_shape = []
# Print out each df and it's shape. For is a loop function. "for every name in tables dictionary, print the shape variables"
for name, df in tables.items():
     raw_data_shape.append({"table_name": name, "rows": df.shape[0], "columns": df.shape[1]})

raw_data_shape_df = pd.DataFrame(raw_data_shape).set_index("table_name")

# Display the results
display(raw_data_shape_df)

,col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,col_10,col_11,col_12,col_13,col_14,col_15,col_16,col_17,col_18,col_19,col_20,col_21
table_name,,,,,,,,,,,,,,,,,,,,,
movies,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
ratings,userId,movieId,rating,timestamp,rating_datetime,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
links,movieId,imdbId,tmdbId,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
credits,tmdbId,director,top_cast,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keywords,tmdbId,keywords_list,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,rows,columns
table_name,,
movies,9082,21
ratings,99810,5
links,9125,3
credits,9082,3
keywords,9082,2


We can confirm a few facts:  
1) **Granularity Match**: The movies, credits, and keywords datasets contain 9,082 rows. The links df contains 43 more rows. It's possible the links table contains additional movieids that are not included in the other tables. There may also be duplicate ids in each table, but it seems these df's row level represents a single movie.  

2) **Ratings Shape**: The ratings df has almost 100k rows. The most popular movies might receive a large portion of these reviews. This will require further analysis.  

3) **Aggregation Required**: Since the ratings row level doesn't match the other tables any joining of the ratings table will require aggregation. We could examine the average rating per movie, distribution of review scores, count of reviews, etc.  

4) **Movie Sales Figures**: The movies dataset contains budget and revenue columns. These will be useful when calculating ROI per movie and sorting by revenue. One thing to consider is highly rated movies might not make the most revenue. Many classic movies such as "Fight Club" and "Donnie Darko" had a low or negative ROI.

---               
## Inspecting Missing,Duplicat, & Unusual Values

Next, we will profile each data set and document issues that may require cleaning in the next project milestone. There are 5 sections, each dedicated to a dataset. 

For each dataset we will:  
    - review data types.  
    - check for missing values, empty strings, and zero values.  
    - ensure no primary key is duplicated.
    - 


---

**Movies**

The movies dataset contains over 9000 movies with 21 columns of details. We will profile the data below using python functions such as .dtypes, nunique, .isna.mean, and .isna.sum. It's more clear if we can view all the details regarding a dataset in a single table output.

In [20]:
# Create a dataframe to store movies data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.abs

# Convert all number columns into true or false so we can count where 0 is true
num_cols = movies.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# Strip text columns nas,blanks to make sure they are really empty.
text_cols = movies.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# 3. Build the DataFrame
movies_profile = pd.DataFrame({
    'dtype': movies.dtypes, 
    'unique_values' : movies.nunique(),
    'missing_rate': round(movies.isna().mean() * 100,2), 
    'missing_count': movies.isna().sum(),
    'zero_count': zero_mask.sum().reindex(movies.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(movies.columns, fill_value=0) * 100,2),
    'empty_count': empty_mask.sum().reindex(movies.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(movies.columns, fill_value=0) * 100,2)
})

print("Movies Profile Table")
display(movies_profile.sort_values(by='missing_rate', ascending=False))


Movies Profile Table


,dtype,unique_values,missing_rate,missing_count,zero_count,zero_rate,empty_count,empty_rate
genres_list,object,1694,0.39,35,0,0.00,0,0.0
primary_genre,object,20,0.39,35,0,0.00,0,0.0
status,object,4,0.02,2,0,0.00,0,0.0
tmdbId,int64,9082,0.00,0,0,0.00,0,0
movieId,int64,9082,0.00,0,0,0.00,0,0
original_title,object,8841,0.00,0,0,0.00,0,0.0
title,object,8809,0.00,0,0,0.00,0,0.0
imdb_id,object,9082,0.00,0,0,0.00,0,0.0
original_language,object,42,0.00,0,0,0.00,0,0.0
revenue,float64,4555,0.00,0,4322,47.59,0,0


In [29]:
movies.head(20)
(movies['revenue'] % 1 > 0).sum()
#movies["primary_genre"].unique()

np.int64(0)

The output table generated above provides some key profiling information:

1) **dtypes**: There are a few things that could be changed.
    - ***release_date*** is an object. It should be converted to a date using pd.to_datetime().
    - ***vote_count*** is set as float64 but it's a whole number so it should be int64.
    - ***revenue*** and ***budget*** represent dollars but are different dtypes. Let's plan on standardizing them as float64. Float64 allows for NaN values, which will prevent them from being included in math calculations.
    - Columns with few unqiue values could be converted to ***category*** data types, but it's not required.

2) **Ratings Shape**: The ratings df has almost 100k rows. The most popular movies might receive a large portion of these reviews. This will require further analysis.  

3) **Aggregation Required**: Since the ratings row level doesn't match the other tables any joining of the ratings table will require aggregation. We could examine the average rating per movie, distribution of review scores, count of reviews, etc.  

4) **Movie Sales Figures**: The movies dataset contains budget and revenue columns. These will be useful when calculating ROI per movie and sorting by revenue. One thing to consider is highly rated movies might not make the most revenue. Many classic movies such as "Fight Club" and "Donnie Darko" had a low or negative ROI.

In [18]:
movies.describe().T

#dup_titles = movies[movies['title'].duplicated(keep=False)]
#display(dup_titles.sort_values('title')[['movieId', 'title', 'release_date']].head(10))


,count,mean,std,min,25%,50%,75%,max
movieId,9082.0,3.096374e+04,4.065969e+04,1.000000,2843.250000,6265.500000,5.602625e+04,1.639490e+05
tmdbId,9082.0,3.871046e+04,6.212341e+04,2.000000,9446.250000,15775.000000,3.901025e+04,4.164370e+05
runtime,9082.0,1.056614e+02,3.035071e+01,0.000000,93.000000,102.000000,1.150000e+02,1.140000e+03
budget,9082.0,1.664789e+07,3.345276e+07,0.000000,0.000000,100000.000000,2.000000e+07,3.800000e+08
revenue,9082.0,4.918728e+07,1.301801e+08,0.000000,0.000000,271608.000000,3.672528e+07,2.787965e+09
popularity,9082.0,7.268905e+00,9.372314e+00,0.000004,2.954521,6.418871,9.861742e+00,5.474883e+02
vote_average,9082.0,6.362178e+00,1.038748e+00,0.000000,5.800000,6.500000,7.000000e+00,1.000000e+01
vote_count,9082.0,4.390509e+02,9.961565e+02,0.000000,29.000000,96.000000,3.657500e+02,1.407500e+04
release_year,9082.0,1.991952e+03,1.938771e+01,1902.000000,1984.000000,1997.000000,2.006000e+03,2.016000e+03


In [ ]:
# Create a dataframe to store movies data types and missing rate/count info.
# Target 'object' columns since that's how text/mixed data is stored.abs

# 1. Numeric masks for zeros
num_cols = movies.select_dtypes(include='number')
zero_mask = (num_cols == 0)

# 2. Text masks for empty/whitespace strings
text_cols = movies.select_dtypes(include=['object'])
empty_mask = text_cols.apply(lambda col: col.dropna().astype(str).str.strip() == '')

# 3. Build the DataFrame
movies_datatypes = pd.DataFrame({
    'dtype': movies.dtypes, 
    'missing_rate': round(movies.isna().mean() * 100,2), 
    'missing_count': movies.isna().sum(),
    
    'zero_count': zero_mask.sum().reindex(movies.columns, fill_value=0),
    'zero_rate': round(zero_mask.mean().reindex(movies.columns, fill_value=0) * 100,2),
    
    'empty_count': empty_mask.sum().reindex(movies.columns, fill_value=0),
    'empty_rate': round(empty_mask.mean().reindex(movies.columns, fill_value=0) * 100,2)
})

print("--- movies Dtypes and Missing Review ---")
display(movies_datatypes.sort_values(by='missing_rate', ascending=False))

# Create a movies missing dataset that lists every movieId with a missing value.
# This will be used later to filter out missing movies from analysis. I could query "not in movies_missing[movieId]"
# I could also make movies_copy = movies - movies_missing['movieID']
movies_missing = movies[movies.isna().any(axis=1)][['movieId','title','release_date','genres_list','primary_genre','status']]
display(movies_missing.sort_values(by = 'status', ascending = True).head())

# This will return the exact count of movies with a budget of 0
zero_budget_count = (movies['budget'] == 0).sum()
zero_revenue_count = (movies['revenue'] == 0).sum()

display(movies[movies['revenue'] == 0])

display(movies['vote_average'].value_counts(normalize = True))
display(movies.sort_values(by='popularity', ascending = False).head(20))

print(f"zero budget count: {zero_budget_count}\nzero revenue count: {zero_revenue_count}")

In [ ]:
## In this code I examine the movies revenue. This project focuses on the primary unit of data, which is movies. This table is most important.
# limit column width so the text doesnt wrap and make the rows wide.
pd.set_option('display.max_colwidth',12)

# Drop only the columns that cause wide text wrapping
movie_cols_to_show = [col for col in movies.columns if col not in ["genres", "production_companies", "production_countries"]]
#display(movie_cols_to_show)

# Display top 3 movies by revenue
display(
    movies.sort_values(by="revenue", ascending=True).head(3)
)

In [ ]:
# Display top 3 movies by release date desc. Release date is string, but sorting still works on it bc it's in numerical order.
display(movies.sort_values(by="release_date", ascending=False)[["movieId", "title", "release_date"]].head(3))

# Convert release_date to actual datetime objects, then sort
movies_sorted = movies.copy()
movies_sorted['release_date'] = pd.to_datetime(movies_sorted['release_date'], errors='coerce')

display(
    movies_sorted.sort_values(by="release_date", ascending=False)[["movieId", "title", "release_date"]].head(3)
)

#Sorting by date has same result. Most recent movie is september 2023.

In [ ]:
display(ratings['movieId'].drop_duplicates())

display(ratings['rating_datetime'].sort_values(ascending=False).iloc[0])
display(ratings['rating_datetime'].sort_values(ascending=True).iloc[0])

display(movies['release_date'].sort_values(ascending=False).iloc[0])
display(movies['release_date'].sort_values(ascending=True).iloc[0])

In [ ]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))

In [ ]:
for name, df in tables.items():
    print(f"\n--- {name} Profile Summary ---")
    
    # 1. Create a summary DataFrame containing dtypes and missing rates
    summary_df = pd.DataFrame({
        'dtype': df.dtypes,
        'missing_rate': df.isna().mean(),
        'missing_count': df.isna().sum()
    })
    
    # 2. Display the combined table
    display(summary_df.sort_values(by='missing_rate', ascending=False))

In [ ]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))

In [ ]:
print("--- Movies ---")
display(movies[movies["movieId"] == 1])

print("--- Ratings ---")
display(ratings[ratings["movieId"] == 1])

print("--- Links ---")
display(links[links["movieId"] == 1])

print("--- Credits ---")
display(credits[credits["tmdbId"] == 862])

print("--- Keywords ---")
display(keywords[keywords["tmdbId"] == 862])

display(movies.sort_values(by="revenue", ascending=False).head(10))

display(movies.sort_values(by="release_date", ascending=False).head(10))

movies[movies.isna().any(axis=1)]

display(movies[movies.isna().any(axis=1) & (movies["revenue"] != 0)])

# Group by movieId to get the average, median, and total rating count
rating_summary = ratings.groupby("movieId").agg(
    rating_avg=("rating", "mean"),
    rating_median=("rating", "median"),
    rating_count=("userId", "count")
).reset_index()

print("--- Merge Rating Summary to Movie df ---")
rating_summary_with_titles = rating_summary.merge(
    movies[["movieId", "title", "vote_average", "vote_count"]], 
    on="movieId", 
    how="left"
)

# Display top 100 with titles
display(rating_summary_with_titles.sort_values(by="rating_count", ascending=False).head(100))